CSU Transparent California


Make sure to download the transparent california zip from the Raw files folder in Google Drive. Upload all csv's from the CSU folder in the zip.

In [ ]:
# Lossless data processing for provided downloaded from Transparent Califronia

import polars as pl

def main():
    print("Scanning CSV files...")

    # Numerical cols
    financial_cols = [
        "base", "overtime", "other", "totalpay", "totalbenefits",
        "totalpaybenefits"
    ]

    df = pl.scan_csv(
        "../data/raw/**/*.csv",
        include_file_paths="filepath"
    )
    # Empty Cols
    df = df.drop(['ual', 'benefitsee', 'benefitser', 'benefitsdc'], strict=False)

    # Helper function to currency text into Ints!
    def clean_currency(col_name):
        return ( pl.col(col_name) .cast(pl.Int32, strict=False))

    df = df.rename(str.lower)

    # Apply typing and data transformations
    df = df.with_columns(
        pl.selectors.string().str.to_lowercase()
    ).with_columns(
        # Get Year From Filename
        pl.col("filepath").str.extract(r"-(\d{4})\.csv$", 1).cast(pl.Int16).alias("year"),

        # Categorical cols
        pl.col("job").cast(pl.Categorical),
        pl.col("department").cast(pl.Categorical),

        # Clean and cast all financial columns
        *[clean_currency(col) for col in financial_cols]
    )

    df = df.drop(["filepath"])

    final_df = df.collect()
    final_df.write_parquet("/content/csu_salaries_2013-2024.parquet", compression="zstd")

    print(f"Processed {final_df.height:,} records.")
    schema = final_df.schema
    for col_name, dtype in schema.items():
        print(f"{col_name}: {dtype}")

if __name__ == "__main__":
    main()

Transparent California Filtered CSU Data

In [ ]:
import re

import pandas as pd

CSU_ROLES = [
    ("supplemental_teaching_pay", r"special programs|summer session|extension|additional employment|music studio"),
    ("ladder_faculty", r"^instructional faculty|^grant-related/specially funded instructional faculty|^department chair|^cma instructional faculty"),
    ("lecturer", r"lecturer"),
    ("clinical_adjunct_faculty", r"^visiting faculty"),
    ("postdoc", r"^research fellow|post-doctoral"),
    ("trainee", r"counselor intern"),
    ("researcher_specialist_librarian", r"librarian|academic-related|coach"),
]


def role_of(title, rules) -> str:
    t = str(title or "").lower().strip()
    return next((role for role, pattern in rules if re.search(pattern, t)), "staff")


csu_salaries = pd.read_parquet("/content/csu_salaries_2013-2024.parquet")

# Make sure pay is numeric before filtering
csu_salaries["base"] = pd.to_numeric(csu_salaries["base"], errors="coerce")

filtered_csu_salaries = csu_salaries.copy()
filtered_csu_salaries["role"] = filtered_csu_salaries["job"].map(lambda t: role_of(t, CSU_ROLES)).astype("string")

# Replace "name" with your actual name-column name if needed
filtered_csu_salaries["name"] = (
    filtered_csu_salaries["name"]
    .str.title()
    .str.replace(r"(?<=\s)[A-Z]\.?(?=\s|$)", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

filtered_csu_salaries.to_parquet(
    "/content/transparent_california_filtered_csu_salaries_2013-2024.parquet",
    index=False
)

No Duplicates in each year for each role

In [ ]:
csu_salaries = pd.read_parquet("/content/transparent_california_filtered_csu_salaries_2013-2024.parquet")

no_dup_filtered_csu_salaries = csu_salaries.drop_duplicates(subset=["year", "name", "role"])

no_dup_filtered_csu_salaries.to_parquet(
    "/content/transparent_california_no_dup_filtered_csu_salaries_2013-2024.parquet",
    index=False
)
